# NP Target Navigator: walkthrough

This notebook runs the same functions the Streamlit app uses, so every number in the README can be reproduced here.

**Before running:** complete the data setup in the README (`clean_chembl.py`, then `fingerprints.py`). The notebook works from the repo root automatically.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")          # run everything from the repo root
sys.path.insert(0, "src")

import pandas as pd
from IPython.display import display

from predict import predict_targets
from admet import get_admet_flags

pd.set_option("display.width", 160)
print("Working directory:", os.getcwd())

## 1. A known natural product: capsaicin

The compound is hidden from the library (`exclude_exact=True`), so the targets come only from its structural analogs.

In [ ]:
CAPSAICIN = "COc1cc(CNC(=O)CCCC/C=C/C(C)C)ccc1O"

res = predict_targets(CAPSAICIN, top_n=10, exclude_exact=True)
print(f"Closest known ligand similarity: {res.attrs['max_sim']:.2f}")
display(res.round(2))

`score` is the similarity-weighted vote from the nearest known ligands. `confidence` is the share of neighbour similarity that supports the target. It is a heuristic, not a probability. `n_neighbors` and `mean_pchembl` show how many known ligands support the hit and how potent they are.

In [ ]:
flags = get_admet_flags(CAPSAICIN)
display(pd.Series(flags, name="capsaicin").to_frame())

## 2. A compound far from the knowledge base

`CNP0169375.1` is a COCONUT natural product whose nearest ChEMBL ligand has similarity of about 0.31. The app warns below 0.4.

In [ ]:
scan = pd.read_csv("docs/coconut_novelty_scan.csv")
row = scan[scan["identifier"] == "CNP0169375.1"].iloc[0]
far_smiles = row["smiles_std"]
print("COCONUT compound:", row["identifier"],
      "| similarity to nearest ChEMBL ligand:", round(row["max_sim_to_chembl"], 3))

res_far = predict_targets(far_smiles, top_n=10, min_sim=0.25, exclude_exact=True)
print(f"Closest known ligand similarity: {res_far.attrs['max_sim']:.2f}")
if res_far.attrs["max_sim"] < 0.4:
    print("LOW CONFIDENCE: far from known ligands, so treat these as weak hypotheses.")
display(res_far.round(2))

## 3. Validation with 95% intervals

Produced by `src/validate.py` and `src/validation_ci.py`. The 0.4-0.7 group is the more realistic guide for natural products.

In [ ]:
print(open("docs/validation_ci.txt").read())

## 4. How far are natural products from the training data?

Random sample of COCONUT compounds from `src/novelty_scan.py`.

In [ ]:
scan = pd.read_csv("docs/coconut_novelty_scan.csv")
s = scan["max_sim_to_chembl"]
print(f"Compounds scanned: {len(s)}")
print(f"Median similarity to nearest ChEMBL ligand: {s.median():.2f}")
for t in (0.3, 0.4, 0.7):
    print(f"Share below {t}: {(s < t).mean():.0%}")

**Limits:** predictions are hypotheses for experimental follow-up. Performance depends on how close a compound is to known ligands, and the sample here is small (about 400 compounds, so roughly +/-4 percentage points).